# Notebook 04: Expansion Model
## SaaS Expansion Analytics — RavenStack Dataset
**Objective:** Build a logistic regression model to predict upgrade likelihood. Use SHAP values to explain individual predictions. Output an expansion readiness score (0-100) per account.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import shap
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    roc_curve,
    confusion_matrix
)
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print("Libraries loaded")

In [ ]:
df = pd.read_csv('../data/processed/account_features.csv')

print(f"Account features loaded: {df.shape}")
print(f"Upgrade rate: {df['has_upgraded'].mean()*100:.1f}%")
print(f"\nClass balance:")
print(df['has_upgraded'].value_counts())

## 1. Feature Selection
Select features for the model based on Notebook 03 signal analysis.
Exclude identifiers, leakage columns, and low-signal features.

In [ ]:
# Categorical features to encode
cat_features = ['plan_tier', 'industry', 'referral_source']

# Numeric features
num_features = [
    # Usage signals — strongest predictors
    'total_usage_count',
    'unique_features_used',
    'total_usage_duration',
    'beta_feature_usage',
    'total_errors',
    # Subscription signals
    'total_mrr',
    'total_seats',
    'n_subscriptions',
    # Support signals
    'n_tickets',
    'avg_satisfaction',
    'n_escalations',
    # Billing signals
    'is_annual',
    'auto_renew'
]

# Encode categorical features
df_model = pd.get_dummies(
    df[cat_features + num_features + ['has_upgraded']],
    columns=cat_features,
    drop_first=False
)

# Convert booleans to int
bool_cols = df_model.select_dtypes(include='bool').columns
df_model[bool_cols] = df_model[bool_cols].astype(int)

print(f"Model dataset: {df_model.shape}")
print(f"Features: {df_model.shape[1] - 1}")
print(f"\nFeature list:")
feature_cols = [c for c in df_model.columns if c != 'has_upgraded']
for f in feature_cols:
    print(f"  {f}")

# Define X and y for model
X = df_model.drop('has_upgraded', axis=1)
y = df_model['has_upgraded'].astype(int)

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Upgrade rate: {y.mean()*100:.1f}%")


## 2. Train / Test Split
80/20 split — stratified to preserve class balance.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set:  {X_train.shape}")
print(f"Test set:      {X_test.shape}")
print(f"\nClass balance in training set:")
print(y_train.value_counts())
print(f"\nUpgrade rate train: {y_train.mean()*100:.1f}%")
print(f"Upgrade rate test:  {y_test.mean()*100:.1f}%")

## 3. Logistic Regression Model

In [ ]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(
                   random_state=42,
                   max_iter=1000,
                   class_weight='balanced'))
])

pipeline.fit(X_train, y_train)

y_pred      = pipeline.predict(X_test)
y_pred_prob = pipeline.predict_proba(X_test)[:, 1]
auc         = roc_auc_score(y_test, y_pred_prob)

print(f"=== Model Performance ===")
print(f"AUC-ROC: {auc:.3f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred,
      target_names=['Not Upgraded', 'Upgraded']))

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fpr, y=tpr,
    mode='lines',
    name=f'ROC Curve (AUC = {auc:.3f})',
    line=dict(color='#3498db', width=2)
))
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode='lines',
    name='Random Classifier',
    line=dict(color='gray', width=1, dash='dash')
))
fig.update_layout(
    title='ROC Curve — Expansion Model',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    height=400,
    font=dict(size=13)
)
fig.show()

## 4. Feature Importance
Which features drive the model's predictions?

In [ ]:
# Extract model and scaler from pipeline
model  = pipeline.named_steps['model']
scaler = pipeline.named_steps['scaler']

importance = pd.DataFrame({
    'feature':     X.columns,
    'coefficient': model.coef_[0]
}).sort_values('coefficient', key=abs, ascending=True)

colors = [
    '#2ecc71' if x > 0 else '#e74c3c'
    for x in importance['coefficient']
]

fig = go.Figure(go.Bar(
    x=importance['coefficient'],
    y=importance['feature'],
    orientation='h',
    marker_color=colors,
    text=[f"{x:.3f}" for x in importance['coefficient']],
    textposition='outside'
))

max_coef = importance['coefficient'].abs().max()
fig.update_layout(
    title='Feature Importance — Logistic Regression Coefficients<br>'
          '<sub>Green = increases upgrade likelihood | '
          'Red = decreases upgrade likelihood</sub>',
    height=550,
    font=dict(size=12),
    xaxis=dict(range=[
        -max_coef * 1.4,
         max_coef * 1.4
    ]),
    xaxis_title='Coefficient'
)
fig.show()

positive = (importance[importance['coefficient'] > 0]
            .sort_values('coefficient', ascending=False)
            .head(5))

negative = (importance[importance['coefficient'] < 0]
            .sort_values('coefficient', ascending=True)
            .head(5))

print("\nTop 5 features INCREASING upgrade likelihood:")
print(positive[['feature', 'coefficient']].to_string(index=False))

print("\nTop 5 features DECREASING upgrade likelihood:")
print(negative[['feature', 'coefficient']].to_string(index=False))

## 5. SHAP Values
SHAP explains individual predictions — not just overall feature importance.
For each account: which features pushed their score up or down?

In [ ]:
# SHAP explainer on training data
X_train_scaled = scaler.transform(X_train)
X_test_scaled  = scaler.transform(X_test)

explainer   = shap.LinearExplainer(
    model,
    X_train_scaled,
    feature_names=X.columns.tolist()
)
shap_values = explainer(X_test_scaled)

print(f"SHAP values computed for {len(X_test)} test accounts")
print(f"Shape: {shap_values.values.shape}")

In [ ]:
# Convert to matplotlib for SHAP plots
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values.values,
    X_test,
    feature_names=X.columns.tolist(),
    show=False
)
plt.title('SHAP Summary — Feature Impact on Upgrade Prediction')
plt.tight_layout()
plt.savefig('../outputs/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print("SHAP summary plot saved")

## 6. Expansion Readiness Score
Convert model probability to a 0-100 score per account.
Apply to ALL active Basic and Pro accounts — our expansion universe.

In [ ]:
# Load expansion universe
expansion_universe = pd.read_csv(
    '../data/processed/expansion_universe.csv'
)

# Build features for expansion universe
accounts_all  = pd.read_csv('../data/raw/ravenstack_accounts.csv')
subscriptions = pd.read_csv('../data/raw/ravenstack_subscriptions.csv')
feature_usage = pd.read_csv('../data/raw/ravenstack_feature_usage.csv')
support       = pd.read_csv('../data/raw/ravenstack_support_tickets.csv')

subscriptions['start_date'] = pd.to_datetime(subscriptions['start_date'])
subscriptions['end_date']   = pd.to_datetime(subscriptions['end_date'])
feature_usage['usage_date'] = pd.to_datetime(feature_usage['usage_date'])
support['submitted_at']     = pd.to_datetime(support['submitted_at'])

active_subs = subscriptions[
    (subscriptions['end_date'].isna()) &
    (subscriptions['mrr_amount'] > 0)
]

# Rebuild features for expansion universe only
eu_ids = set(expansion_universe['account_id'])

sub_features = (active_subs[active_subs['account_id'].isin(eu_ids)]
    .groupby('account_id')
    .agg(
        total_mrr       =('mrr_amount', 'sum'),
        avg_mrr         =('mrr_amount', 'mean'),
        max_mrr         =('mrr_amount', 'max'),
        total_seats     =('seats', 'sum'),
        avg_seats       =('seats', 'mean'),
        n_subscriptions =('subscription_id', 'count'),
        has_upgraded    =('upgrade_flag', 'max'),
        has_downgraded  =('downgrade_flag', 'max'),
        is_annual       =('billing_frequency',
                          lambda x: (x == 'annual').any()),
        auto_renew      =('auto_renew_flag', 'max')
    ).reset_index())

usage_with_account = feature_usage.merge(
    active_subs[['subscription_id', 'account_id']],
    on='subscription_id', how='inner'
)
usage_features = (usage_with_account[
    usage_with_account['account_id'].isin(eu_ids)]
    .groupby('account_id')
    .agg(
        total_usage_count    =('usage_count', 'sum'),
        avg_usage_count      =('usage_count', 'mean'),
        total_usage_duration =('usage_duration_secs', 'sum'),
        unique_features_used =('feature_name', 'nunique'),
        total_errors         =('error_count', 'sum'),
        beta_feature_usage   =('is_beta_feature', 'sum')
    ).reset_index())

support_features = (support[support['account_id'].isin(eu_ids)]
    .groupby('account_id')
    .agg(
        n_tickets            =('ticket_id', 'count'),
        avg_satisfaction     =('satisfaction_score', 'mean'),
        avg_resolution_hours =('resolution_time_hours', 'mean'),
        n_escalations        =('escalation_flag', 'sum'),
        n_urgent             =('priority',
                               lambda x: (x == 'urgent').sum())
    ).reset_index())

df_eu = (expansion_universe
    .merge(sub_features,     on='account_id', how='left')
    .merge(usage_features,   on='account_id', how='left')
    .merge(support_features, on='account_id', how='left')
)

# Fill nulls
for col in ['total_usage_count', 'unique_features_used',
            'total_errors', 'beta_feature_usage', 'n_tickets',
            'n_escalations', 'n_urgent']:
    df_eu[col] = df_eu[col].fillna(0)

df_eu['avg_satisfaction']     = df_eu['avg_satisfaction'].fillna(
    df_eu['avg_satisfaction'].median()
)
df_eu['avg_resolution_hours'] = df_eu['avg_resolution_hours'].fillna(
    df_eu['avg_resolution_hours'].median()
)

print(f"Expansion universe features built: {df_eu.shape}")

In [ ]:
# Encode categoricals — match training columns exactly
df_eu_model = pd.get_dummies(
    df_eu[cat_features + num_features],
    columns=cat_features,
    drop_first=False
)

bool_cols = df_eu_model.select_dtypes(include='bool').columns
df_eu_model[bool_cols] = df_eu_model[bool_cols].astype(int)

# Align columns with training data
df_eu_model = df_eu_model.reindex(
    columns=X.columns, fill_value=0
)

# Scale and predict via pipeline
eu_probs = pipeline.predict_proba(df_eu_model)[:, 1]

# Convert to 0-100 score
df_eu['upgrade_probability']       = eu_probs
df_eu['expansion_readiness_score'] = (eu_probs * 100).round(1)

# Score band
df_eu['readiness_band'] = pd.cut(
    df_eu['expansion_readiness_score'],
    bins=[0, 25, 50, 75, 100],
    labels=['Low', 'Medium', 'High', 'Very High']
)

# ── SHAP values for expansion universe ───────────────────────
scaler_eu     = pipeline.named_steps['scaler']
X_eu_scaled   = scaler_eu.transform(df_eu_model)
shap_eu       = explainer(X_eu_scaled)

shap_df = pd.DataFrame(
    shap_eu.values,
    columns=X.columns
)

def top_reasons(row, n=3):
    """Extract top 3 SHAP drivers for each account"""
    top = row.abs().nlargest(n)
    reasons = []
    for feat in top.index:
        direction = '+' if row[feat] > 0 else '-'
        reasons.append(
            f"{feat} ({direction}{abs(row[feat]):.2f})"
        )
    return ' | '.join(reasons)

df_eu['top_shap_reasons'] = shap_df.apply(top_reasons, axis=1)

print("=== Expansion Readiness Score Distribution ===")
print(df_eu['readiness_band'].value_counts().sort_index())
print(f"\nAvg score: {df_eu['expansion_readiness_score'].mean():.1f}")
print(f"Top score: {df_eu['expansion_readiness_score'].max():.1f}")
print(f"Low score: {df_eu['expansion_readiness_score'].min():.1f}")

print(f"\nSample SHAP reasons (top 5 accounts):")
print(df_eu.nlargest(5, 'expansion_readiness_score')[
    ['account_id', 'expansion_readiness_score', 'top_shap_reasons']
].to_string(index=False))

In [ ]:
fig = go.Figure(go.Histogram(
    x=df_eu['expansion_readiness_score'],
    nbinsx=20,
    marker_color='#3498db',
    opacity=0.8
))
fig.update_layout(
    title='Expansion Readiness Score Distribution',
    xaxis_title='Expansion Readiness Score (0-100)',
    yaxis_title='Number of Accounts',
    height=400,
    font=dict(size=13)
)
fig.show()

In [ ]:
df_scored = df_eu[[
    'account_id', 'account_name', 'industry',
    'country', 'plan_tier', 'referral_source',
    'total_mrr', 'total_seats',
    'total_usage_count', 'unique_features_used',
    'avg_satisfaction', 'n_escalations',
    'upgrade_probability', 'expansion_readiness_score',
    'readiness_band', 'top_shap_reasons'        # ← added
]].sort_values('expansion_readiness_score', ascending=False)

df_scored.to_csv(
    '../data/processed/scored_accounts.csv', index=False
)
print(f"Saved scored_accounts.csv — {len(df_scored)} accounts")
print(f"\nTop 10 expansion targets:")
print(df_scored.head(10)[[
    'account_name', 'industry', 'plan_tier',
    'total_mrr', 'expansion_readiness_score',
    'readiness_band', 'top_shap_reasons'        # ← added
]].to_string(index=False))

## 7. Key Findings

**Model performance — AUC 0.618:**
The model has modest predictive power on this synthetic dataset.
AUC of 0.618 is better than random (0.5) but below the 0.7
threshold for a reliable classifier. On real CRM data with
genuine behavioural signals and larger sample sizes,
performance would likely improve significantly.
The SHAP explanations remain actionable regardless of score.

**Usage duration is the dominant expansion signal:**
total_usage_duration has the strongest positive coefficient (0.570).
Accounts that spend more time in the product are most likely
to upgrade — depth of engagement matters more than frequency.

**Multicollinearity between usage metrics:**
total_usage_count shows a negative coefficient (-0.496) despite
being positively correlated with upgrade in Notebook 03.
This is a classic multicollinearity effect — duration and count
are highly correlated, causing the model to assign opposite signs.
Duration is the cleaner signal; count adds noise.

**Billing and satisfaction features are near-zero:**
auto_renew (0.000), is_annual (-0.029), avg_satisfaction (0.015)
contribute almost nothing to predictions.
In this synthetic dataset, behavioural usage signals
dominate over account configuration signals.

**Escalations are a positive signal (counterintuitive):**
n_escalations has a positive coefficient (0.220).
Accounts that escalate support tickets are more engaged
with the product — they care enough to escalate.
This may reflect an engaged power user base
more likely to expand than disengage.